In [4]:
# Install if needed (run once)
!pip install -q langchain-text-splitters sentence-transformers chromadb

import pandas as pd
import numpy as np
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import uuid
import shutil
import time

# --- Fix: Set project root correctly ---
# Since this notebook is in the "notebooks/" folder, go one level up
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "filtered_complaints.csv"
VECTOR_STORE_PATH = PROJECT_ROOT / "vector_store" / "chroma_db"

print(f"Data path: {PROCESSED_DATA_PATH}")
print(f"Exists? {PROCESSED_DATA_PATH.exists()}")

# --- Configuration ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
SAMPLE_SIZE = 2000          # Small for speed
PRODUCT_COL = "Product"

def stratified_sample(df, sample_size, stratum_col):
    strata = df[stratum_col].value_counts(normalize=True)
    sampled_dfs = []
    for stratum, proportion in strata.items():
        n = int(round(proportion * sample_size))
        if n <= 0:
            continue
        stratum_df = df[df[stratum_col] == stratum]
        if len(stratum_df) < n:
            sampled = stratum_df.sample(n=n, replace=True, random_state=42)
        else:
            sampled = stratum_df.sample(n=n, random_state=42)
        sampled_dfs.append(sampled)
    result = pd.concat(sampled_dfs).reset_index(drop=True)
    if len(result) > sample_size:
        result = result.sample(n=sample_size, random_state=42)
    return result

# Load data
print("Loading data...")
df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Total: {len(df)}")

# Sample
print("Sampling...")
df_sample = stratified_sample(df, SAMPLE_SIZE, PRODUCT_COL)
print(f"Sampled: {len(df_sample)}")
print(df_sample[PRODUCT_COL].value_counts())

# Chunking
print("Chunking...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []
for idx, row in df_sample.iterrows():
    narrative = row.get("cleaned_narrative", "")
    if not isinstance(narrative, str) or len(narrative.strip()) == 0:
        continue
    text_chunks = text_splitter.split_text(narrative)
    for i, chunk in enumerate(text_chunks):
        chunks.append({
            "complaint_id": row.get("Complaint ID", idx),
            "product": row[PRODUCT_COL],
            "chunk_text": chunk,
            "chunk_index": i,
            "total_chunks": len(text_chunks)
        })
print(f"Chunks: {len(chunks)}")

# Embeddings
print("Generating embeddings...")
model = SentenceTransformer(EMBEDDING_MODEL)
chunk_texts = [c["chunk_text"] for c in chunks]
batch_size = 64
all_embeddings = []
for i in range(0, len(chunk_texts), batch_size):
    batch = chunk_texts[i:i+batch_size]
    batch_emb = model.encode(batch, show_progress_bar=True)
    all_embeddings.extend(batch_emb)
print(f"Embedding dim: {len(all_embeddings[0])}")

# Store in ChromaDB
print("Building ChromaDB...")
if VECTOR_STORE_PATH.exists():
    shutil.rmtree(VECTOR_STORE_PATH)
client = chromadb.PersistentClient(path=str(VECTOR_STORE_PATH))
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)
collection = client.get_or_create_collection(
    name="complaints",
    embedding_function=embedding_fn
)

ids = [str(uuid.uuid4()) for _ in range(len(chunks))]
documents = [c["chunk_text"] for c in chunks]
metadatas = [
    {
        "complaint_id": str(c["complaint_id"]),
        "product": c["product"],
        "chunk_index": c["chunk_index"],
        "total_chunks": c["total_chunks"]
    }
    for c in chunks
]

batch_insert = 1000
for i in range(0, len(chunks), batch_insert):
    end = min(i + batch_insert, len(chunks))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        metadatas=metadatas[i:end]
    )
    print(f"Inserted {end}/{len(chunks)}")

print("✅ Done! Vector store saved.")

Data path: C:\Users\user\OneDrive\Desktop\Project\KAIM\credittrust-complaint-rag\credittrust-complaint-rag\data\processed\filtered_complaints.csv
Exists? True
Loading data...
Total: 82164
Sampling...
Sampled: 2000
Product
Credit card        1964
Money transfers      36
Name: count, dtype: int64
Chunking...
Chunks: 5852
Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dim: 384
Building ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Inserted 1000/5852
Inserted 2000/5852
Inserted 3000/5852
Inserted 4000/5852
Inserted 5000/5852
Inserted 5852/5852
✅ Done! Vector store saved.
